# 00 — Preflight

Validates runtime, dependencies, inputs, licences, splits, labels, IDs and tensor
shapes. **Refuses to discover or load the reserved `internal_test` partition.**

Run this first. If any check fails, stop — do not run the later notebooks.


In [ ]:
import os, sys, json, time, hashlib, platform
from pathlib import Path
WORK = Path('/kaggle/working'); INP = Path('/kaggle/input')
CODE = next((p for p in INP.glob('*/src/guards.py')), None)
if CODE is None:
    raise SystemExit('attach the code dataset; see KAGGLE_INPUT_CHECKLIST.md')
sys.path.insert(0, str(CODE.parent))
import guards as G
print('code from', CODE.parent)


In [ ]:
# --- runtime
import torch
info = {'python': platform.python_version(), 'torch': torch.__version__,
        'cuda': torch.cuda.is_available(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'internet': None}
if torch.cuda.is_available():
    f, t = torch.cuda.mem_get_info()
    info['vram_free_gb'] = round(f/1e9, 2); info['vram_total_gb'] = round(t/1e9, 2)
try:
    import socket; socket.create_connection(('huggingface.co', 443), 5).close()
    info['internet'] = True
except Exception:
    info['internet'] = False
print(json.dumps(info, indent=2))
if not info['cuda']:
    print('WARNING: no GPU. Enable an accelerator for notebook 02.')


In [ ]:
# --- the locked partition must be unreachable
import pytest
ok = False
try:
    G.assert_path_allowed(INP / 'anything' / 'internal_test' / 'x.png')
except G.LockedPartitionError:
    ok = True
assert ok, 'the locked-partition guard is not active'
print('locked-partition guard: ACTIVE')
# and prove no attached input exposes it
leaks = [str(p) for p in INP.rglob('*internal*test*')]
assert not leaks, 'an attached dataset contains locked data: %s' % leaks[:3]
print('no attached input exposes the locked partition')


In [ ]:
# --- manifest and licence flags
man = next((p for p in INP.glob('*/manifest.json')), None)
if man is None:
    print('NO manifest.json found. Fill manifest_template.json and attach it.')
else:
    M = json.loads(Path(man).read_text())
    print(json.dumps(M.get('licences', {}), indent=2))
    blocked = [k for k, v in M.get('licences', {}).items()
               if str(v.get('kaggle_use_permitted')).lower() != 'yes']
    if blocked:
        print('BLOCKED components (unknown == blocked):', blocked)
    else:
        print('all declared components cleared for Kaggle use')


In [ ]:
# --- splits, labels, IDs
import pandas as pd
c01 = next((p for p in INP.rglob('provisional_c01.csv')), None)
if c01:
    df = pd.read_csv(c01, usecols=['image_id', 'split'])
    usable, n_locked = G.drop_locked_rows(df)
    print('usable splits:', usable.split.value_counts().to_dict())
    print('locked rows excluded (count only):', n_locked)
else:
    print('provisional_c01.csv not attached')
cfg = next((p for p in INP.glob('*/configs/target_label.yaml')), None)
if cfg:
    import yaml; tgt = yaml.safe_load(Path(cfg).read_text())
    G.assert_target_mapping(tgt); print('target mapping OK:', tgt['description'])
else:
    print('target_label.yaml not attached - notebook 02 will refuse to train')


In [ ]:
# --- image availability and checksums
imgs = {}
for name in ('vietnam_img512', 'vietnam_img512_lungs'):
    d = next((p for p in INP.rglob(name) if p.is_dir()), None)
    imgs[name] = len(list(d.glob('*.png'))) if d else 0
print('image counts:', imgs)
if not any(imgs.values()):
    print('NO images attached. Notebooks 01 and 02 can only run their preflight.')


In [ ]:
# --- synthetic shape self-test (no weights, no data)
import numpy as np
for dim, grid in ((768, 37), (1024, 32)):
    G.assert_shape(np.zeros((4, dim)), (4, dim), 'global')
    G.assert_shape(np.zeros((4, grid*grid, dim)), (4, grid*grid, dim), 'regional')
print('shape assertions OK')
(WORK / 'preflight.json').write_text(json.dumps(
    {'runtime': info, 'images': imgs}, indent=2, default=str))
print('wrote', WORK / 'preflight.json')
